In [46]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [47]:
team_with_spreads = pd.read_csv('./Completed_Dataframes/team_with_spreads.csv')
team_with_spreads

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,MATCHUP,WL,MIN,FGM,FGA,...,Close,ML,2H,Year,SERIES_ID,SERIES_WINS,GAME_KEY,OPPONENT_ABBREVIATION,OPPONENT_SERIES_WINS,SERIES_STANDING
0,42021,1610612737,ATL,Atlanta Hawks,42100101,ATL @ MIA,L,240,29,75,...,220.5,225,1,2022,ATL_vs_MIA_42021,0,ATL_vs_MIA_42021_2022-04-17,MIA,1,0–1
1,42021,1610612748,MIA,Miami Heat,42100101,MIA vs. ATL,W,240,43,82,...,6.5,-265,107.5,2022,ATL_vs_MIA_42021,1,ATL_vs_MIA_42021_2022-04-17,ATL,0,1–0
2,42021,1610612737,ATL,Atlanta Hawks,42100102,ATL @ MIA,L,240,41,87,...,221.0,270,107.5,2022,ATL_vs_MIA_42021,0,ATL_vs_MIA_42021_2022-04-19,MIA,2,0–2
3,42021,1610612748,MIA,Miami Heat,42100102,MIA vs. ATL,W,240,38,79,...,8.0,-330,4.5,2022,ATL_vs_MIA_42021,2,ATL_vs_MIA_42021_2022-04-19,ATL,0,2–0
4,42021,1610612737,ATL,Atlanta Hawks,42100103,ATL vs. MIA,W,240,41,80,...,222.0,105,108.5,2022,ATL_vs_MIA_42021,1,ATL_vs_MIA_42021_2022-04-22,MIA,2,1–2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,42017,1610612761,TOR,Toronto Raptors,41700104,TOR @ WAS,L,240,34,79,...,1.5,-125,105.5,2018,TOR_vs_WAS_42017,2,TOR_vs_WAS_42017_2018-04-22,WAS,2,2–2
666,42017,1610612761,TOR,Toronto Raptors,41700105,TOR vs. WAS,W,240,38,82,...,7,-330,4.5,2018,TOR_vs_WAS_42017,3,TOR_vs_WAS_42017_2018-04-25,WAS,2,3–2
667,42017,1610612764,WAS,Washington Wizards,41700105,WAS @ TOR,L,240,37,90,...,216,260,107,2018,TOR_vs_WAS_42017,2,TOR_vs_WAS_42017_2018-04-25,TOR,3,2–3
668,42017,1610612764,WAS,Washington Wizards,41700106,WAS vs. TOR,L,240,32,79,...,214.5,120,106,2018,TOR_vs_WAS_42017,2,TOR_vs_WAS_42017_2018-04-27,TOR,4,2–4


In [48]:
df = team_with_spreads.copy()

In [49]:
# Replace "pk" with 0.0 in relevant spread columns
import re
import numpy as np

def clean_spread_value(x):
    if isinstance(x, str):
        x = x.strip().lower()
        if x in ['pk', 'pick', "pick'em", 'pickem']:
            return 0.0
        match = re.match(r'^(-?\d*\.?\d+)', x)
        if match:
            return float(match.group(1))
        return np.nan  # fallback for anything else
    return x

spread_columns = ['Open', 'Close', 'ML', '2H']
for col in spread_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_spread_value)

In [50]:
df['WL_BINARY'] = df['WL'].map({'W': 1, 'L': 0})

In [51]:
#  Select only pregame features
features = [
    'TEAM_ABBREVIATION', 'OPPONENT_ABBREVIATION', 'HOME_AWAY',
    'Open', 'Close', 'ML', '2H',
    'SERIES_WINS', 'OPPONENT_SERIES_WINS'
]

X = df[features]
y = df['WL_BINARY']

In [52]:
# Identify types of features
categorical_features = ['TEAM_ABBREVIATION', 'OPPONENT_ABBREVIATION', 'HOME_AWAY']
numeric_features = ['Open', 'Close', 'ML', '2H', 'SERIES_WINS', 'OPPONENT_SERIES_WINS']


In [53]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='mean'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [54]:
#  Build full pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [55]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [56]:
# Train the model
clf.fit(X_train, y_train)

/opt/anaconda3/envs/dev/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', SimpleImputer(),
                                                  ['Open', 'Close', 'ML', '2H',
                                                   'SERIES_WINS',
                                                   'OPPONENT_SERIES_WINS']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['TEAM_ABBREVIATION',
                                                   'OPPONENT_ABBREVIATION',
                                                   'HOME_AWAY'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [57]:
# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8208955223880597

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82        67
           1       0.81      0.84      0.82        67

    accuracy                           0.82       134
   macro avg       0.82      0.82      0.82       134
weighted avg       0.82      0.82      0.82       134



In [61]:
# Get coefficients
coefficients = model.coef_[0]  # Binary classification = one set of coefficients

# Create a dataframe
import pandas as pd
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', key=abs, ascending=False)

# Display top features by absolute coefficient value
print(coef_df.head(10))

                           Feature  Coefficient
4                 num__SERIES_WINS     2.056982
5        num__OPPONENT_SERIES_WINS    -1.939913
45  cat__OPPONENT_ABBREVIATION_LAC    -1.130108
62             cat__HOME_AWAY_Away    -1.107237
17      cat__TEAM_ABBREVIATION_LAC     0.970672
63             cat__HOME_AWAY_Home     0.930877
33      cat__TEAM_ABBREVIATION_WAS     0.891458
48  cat__OPPONENT_ABBREVIATION_MIA     0.811192
7       cat__TEAM_ABBREVIATION_BKN    -0.725187
61  cat__OPPONENT_ABBREVIATION_WAS    -0.723494


In [64]:
# Create a single new game input
new_game = pd.DataFrame([{
    'TEAM_ABBREVIATION': 'HOU',
    'OPPONENT_ABBREVIATION': 'GSW',
    'HOME_AWAY': 'AWAY',
    'Open': 4.5,
    'Close': 5.5,
    'ML': +195,
    '2H': 4,
    'SERIES_WINS': 2,
    'OPPONENT_SERIES_WINS': 3
}])

In [65]:
# Predict
prediction = clf.predict(new_game)
probability = clf.predict_proba(new_game)

print("Predicted Win (1) or Loss (0):", prediction[0])
print("Probability of Win:", probability[0][1])

Predicted Win (1) or Loss (0): 0
Probability of Win: 0.05423816920600765
